In [1]:
# !/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [8]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_001_mae.txt')

restart=None
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)

if args.restart is None:
    # generate "unique" id for the run (very unlikely that two runs will have the same ID)
    checkpoint = None
    latest_checkpoint = 0
    step = 0
    restore = False
    data_split_indices = None
    # restarts run from latest checkpoint
else:
    directory = args.restart  # load directory name
    # load latest checkpoint
    checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
    checkpoint = torch.load(os.path.join(
        checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
    latest_checkpoint = checkpoint['step']
    model_code = checkpoint['ID']  # load ID
    if args.fix_arguments:
        for arg in vars(checkpoint['args']):
            if arg in hyperparam_args:
                print('loading hyperparam arg', arg)
                setattr(args, arg, getattr(checkpoint['args'], arg))
    step = checkpoint['step']
    restore = True
    data_split_indices = checkpoint['data_split_indices']

# no restart directory specified

print('args use gpu', args.use_gpu)
args.use_gpu = False
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

args.normalize=2
args.num_features=128
args.num_basis_functions=32

args.verbose = 0
args.use_gpu = False
args.radii_adjust = True 
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose,
                           radii_adjust=args.radii_adjust)

model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
args use gpu True
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7f6fdeaf0048>, level=2)
level 2
finished init
cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f6fdeaf0598>
conversions out <function kcal_to_kcal at 0x7f6fdeaf0598>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1

In [9]:
sample = dataset.get_properties(5)
# results = model.density_repr_model[0](sample)
results = model(sample)

fs norm before: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

ys norm: [1.230926513671875, 1.5160536766052246]
fs norm: [1.230926513671875, 1.5160536766052246, 0.0, 0.0, 0.0, 0.0]

fs norm before: [1.230926513671875, 1.5160536766052246, 0.0, 0.0, 0.0, 0.0]

ys norm: [1.1291526556015015, 1.3374648094177246, 1.0073449611663818, 1.2424957752227783]
fs norm: [1.4086097478866577, 1.583588719367981, 1.0073449611663818, 1.2424957752227783, 0.0, 0.0]

fs norm before: [1.4086097478866577, 1.583588719367981, 1.0073449611663818, 1.2424957752227783, 0.0, 0.0]

ys norm: [1.1666042804718018, 1.1167912483215332, 1.0180964469909668, 1.1090055704116821, 1.1632742881774902, 1.2494595050811768]
fs norm: [1.391601800918579, 1.6071816682815552, 1.1728835105895996, 1.3798258304595947, 1.1632742881774902, 1.2494595050811768]



In [15]:
1/0.0029,

(344.82758620689657,)